In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import os
import re

try:
    import easyocr
    USE_EASYOCR = True
except ImportError:
    USE_EASYOCR = False
    try:
        import pytesseract
    except ImportError:
        pass

os.makedirs("medicine_dataset", exist_ok=True)

def create_sample_medicine_label(filename, med_name, dosage, instructions, expiry):
    img = Image.new('RGB', (600, 350), color=(245, 248, 250))
    d = ImageDraw.Draw(img)
    d.rectangle([10, 10, 590, 340], outline=(0, 102, 204), width=5)
    d.rectangle([10, 10, 590, 75], fill=(0, 102, 204))
    d.text((25, 30), "Rx PHARMACY - PRESCRIPTION LABEL", fill=(255, 255, 255))
    d.text((25, 100), f"MEDICATION: {med_name}", fill=(0, 0, 0))
    d.text((25, 145), f"DOSAGE: {dosage}", fill=(200, 0, 0))
    d.text((25, 190), f"DIRECTIONS: {instructions}", fill=(0, 0, 0))
    d.text((25, 245), f"EXPIRY DATE: {expiry}", fill=(50, 50, 50))
    d.text((25, 295), "KEEP OUT OF REACH OF CHILDREN", fill=(150, 0, 0))
    img.save(filename)

samples = [
    ("medicine_dataset/med_1.png", "AMOXICILLIN 500MG", "1 CAPSULE", "TAKE BY MOUTH EVERY 8 HOURS FOR 7 DAYS", "12/2027"),
    ("medicine_dataset/med_2.png", "IBUPROFEN 400MG", "1 TABLET", "TAKE WITH FOOD EVERY 6 HOURS AS NEEDED FOR PAIN", "08/2028"),
    ("medicine_dataset/med_3.png", "METFORMIN 850MG", "1 TABLET", "TAKE TWICE DAILY WITH MEALS", "05/2027")
]

for path, name, dose, instr, exp in samples:
    create_sample_medicine_label(path, name, dose, instr, exp)

def preprocess_label_image(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
    _, thresh = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return img, thresh

def extract_text_from_label(image_path):
    orig, processed = preprocess_label_image(image_path)
    if USE_EASYOCR:
        reader = easyocr.Reader(['en'], gpu=False, verbose=False)
        results = reader.readtext(orig, detail=0)
        raw_text = "\n".join(results)
    else:
        try:
            raw_text = pytesseract.image_to_string(processed)
        except Exception:
            raw_text = ""
    return orig, processed, raw_text

def parse_cognitive_medicine_info(raw_text):
    info = {
        "Medication": "Unknown",
        "Dosage": "Unknown",
        "Directions": "Unknown",
        "Expiry": "Unknown",
        "Warnings": []
    }
    lines = [line.strip() for line in raw_text.split('\n') if line.strip()]
    for line in lines:
        upper = line.upper()
        if "MEDICATION" in upper or any(unit in upper for unit in ["MG", "ML", "MCG"]):
            if info["Medication"] == "Unknown":
                info["Medication"] = re.sub(r'(?i)MEDICATION:\s*', '', line)
        if "DOSAGE" in upper or "CAPSULE" in upper or "TABLET" in upper:
            if "TAKE" not in upper and info["Dosage"] == "Unknown":
                info["Dosage"] = re.sub(r'(?i)DOSAGE:\s*', '', line)
        if "DIRECTIONS" in upper or "TAKE" in upper or "EVERY" in upper:
            info["Directions"] = re.sub(r'(?i)DIRECTIONS:\s*', '', line)
        if "EXP" in upper or re.search(r'\d{2}/\d{4}', upper):
            info["Expiry"] = re.sub(r'(?i)EXPIRY DATE:\s*', '', line)
        if "CHILDREN" in upper or "WARNING" in upper or "CAUTION" in upper:
            info["Warnings"].append(line)
    return info

def cognitive_medicine_assistant(image_path):
    orig, processed, raw_text = extract_text_from_label(image_path)
    structured_data = parse_cognitive_medicine_info(raw_text)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Captured Bottle Label")
    axes[0].axis("off")
    
    axes[1].imshow(processed, cmap='gray')
    axes[1].set_title("Preprocessed Vision Input")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    
    print("=== COGNITIVE ASSISTANT OUTPUT ===")
    for k, v in structured_data.items():
        print(f"{k}: {v}")
    
    speech_prompt = (
        f"Audio Output: This bottle contains {structured_data['Medication']}. "
        f"The recommended dosage is {structured_data['Dosage']}. "
        f"Instructions: {structured_data['Directions']}. "
        f"Expiration date is {structured_data['Expiry']}."
    )
    print(speech_prompt)
    print("=" * 34 + "\n")

for path, _, _, _, _ in samples:
    cognitive_medicine_assistant(path)

In [ ]:
# Q2: Emotion-aware Cognitive Assistant using Facial Expression Detection

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

EMOTION_LABELS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

def generate_synthetic_fer_dataset(num_samples=1400, img_size=48):
    X = np.zeros((num_samples, img_size, img_size, 1), dtype=np.float32)
    y = np.zeros(num_samples, dtype=np.int32)
    for i in range(num_samples):
        label = i % len(EMOTION_LABELS)
        y[i] = label
        face = np.ones((img_size, img_size), dtype=np.uint8) * 180
        cv2.circle(face, (24, 24), 20, 220, -1)
        cv2.circle(face, (17, 18), 2, 40, -1)
        cv2.circle(face, (31, 18), 2, 40, -1)
        
        if label == 0:
            cv2.line(face, (13, 13), (20, 16), 30, 2)
            cv2.line(face, (35, 13), (28, 16), 30, 2)
            cv2.line(face, (18, 34), (30, 34), 30, 2)
        elif label == 1:
            cv2.line(face, (14, 15), (20, 15), 30, 1)
            cv2.ellipse(face, (24, 35), (6, 3), 0, 180, 360, 30, 2)
        elif label == 2:
            cv2.circle(face, (17, 18), 3, 20, 1)
            cv2.circle(face, (31, 18), 3, 20, 1)
            cv2.ellipse(face, (24, 34), (5, 4), 0, 0, 360, 30, 1)
        elif label == 3:
            cv2.ellipse(face, (24, 30), (8, 5), 0, 0, 180, 30, 2)
        elif label == 4:
            cv2.ellipse(face, (24, 36), (7, 4), 0, 180, 360, 30, 2)
        elif label == 5:
            cv2.line(face, (14, 12), (20, 12), 30, 1)
            cv2.line(face, (28, 12), (34, 12), 30, 1)
            cv2.circle(face, (24, 34), 4, 30, 2)
        else:
            cv2.line(face, (18, 33), (30, 33), 40, 2)
            
        noise = np.random.normal(0, 8, (img_size, img_size))
        face = np.clip(face + noise, 0, 255).astype(np.float32) / 255.0
        X[i, :, :, 0] = face
    return X, to_categorical(y, num_classes=len(EMOTION_LABELS))

X, y = generate_synthetic_fer_dataset(num_samples=1400)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def build_emotion_cnn(input_shape=(48, 48, 1), num_classes=7):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        Flatten(),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

emotion_model = build_emotion_cnn()
history = emotion_model.fit(X_train, y_train, epochs=12, batch_size=32, validation_split=0.1, verbose=0)
loss, acc = emotion_model.evaluate(X_test, y_test, verbose=0)
print(f"Emotion Model Validation Accuracy: {acc * 100:.2f}%")

def cognitive_emotion_response(emotion, confidence):
    responses = {
        'Angry': "I detect frustration or anger. Let's pause, take a slow deep breath, and break down your task into smaller steps.",
        'Disgust': "You seem uncomfortable or displeased. Would you like me to adjust the current interface or switch topics?",
        'Fear': "You appear anxious or stressed. Everything is under control—let me guide you through this step-by-step.",
        'Happy': "You're in a great mood! Let's channel this positive energy into tackling your primary goals today.",
        'Sad': "I sense you might be feeling down. Take things at your own pace, and let me know how I can lighten your workload.",
        'Surprise': "Something unexpected caught your attention! Would you like more details or clarification on what just happened?",
        'Neutral': "You look calm and focused. Ready whenever you are to proceed with your workflow."
    }
    return f"[{emotion} ({confidence*100:.1f}%)] -> Assistant: {responses.get(emotion, '')}"

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def process_frame_with_cognitive_assistant(frame_bgr, gray_face_48=None):
    display_frame = frame_bgr.copy()
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(30, 30))
    
    if len(faces) == 0 and gray_face_48 is not None:
        roi = gray_face_48.reshape(1, 48, 48, 1)
        preds = emotion_model.predict(roi, verbose=0)[0]
        idx = np.argmax(preds)
        emotion = EMOTION_LABELS[idx]
        conf = preds[idx]
        
        plt.figure(figsize=(3, 3))
        plt.imshow(gray_face_48.squeeze(), cmap='gray')
        plt.title(f"{emotion} ({conf*100:.1f}%)")
        plt.axis('off')
        plt.show()
        print(cognitive_emotion_response(emotion, conf))
        return

    for (x, y_coord, w, h) in faces:
        roi_gray = gray[y_coord:y_coord+h, x:x+w]
        roi_resized = cv2.resize(roi_gray, (48, 48)).astype('float32') / 255.0
        roi_input = np.expand_dims(np.expand_dims(roi_resized, -1), 0)
        
        preds = emotion_model.predict(roi_input, verbose=0)[0]
        idx = np.argmax(preds)
        emotion = EMOTION_LABELS[idx]
        conf = preds[idx]
        
        cv2.rectangle(display_frame, (x, y_coord), (x+w, y_coord+h), (0, 255, 0), 2)
        cv2.putText(display_frame, f"{emotion}: {conf*100:.1f}%", (x, y_coord-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        print(cognitive_emotion_response(emotion, conf))

    plt.figure(figsize=(5, 5))
    plt.imshow(cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

for sample_idx in [0, 3, 4]:
    test_face = X_test[sample_idx]
    dummy_frame = cv2.cvtColor((test_face.squeeze() * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    process_frame_with_cognitive_assistant(dummy_frame, gray_face_48=test_face)